# บทที่ 3 · pandas และฐานข้อมูล PostgreSQL

งานหลักของ data engineer คือ **ดึงข้อมูลจากที่ต่าง ๆ มาเป็นตาราง แล้วจัดการต่อ**

**เรื่องสมมติของคอร์สนี้:** เราเป็นทีมข้อมูลของร้านค้าออนไลน์ (ร้านเดียวกับ session 3)
- ยอดขายจาก **เว็บและแอป** อยู่ใน **ฐานข้อมูล PostgreSQL** ของร้าน (ข้อมูลชุดเดียวกับ MySQL ใน session 3)
- ร้านขายบน **Shopee / Lazada / TikTok Shop** ด้วย แต่ละวัน marketplace ส่ง **ไฟล์ CSV** ยอดขายมาให้

บทนี้: **pandas: สร้าง/อ่าน/กรอง/สรุป/รวมตาราง · ต่อ PostgreSQL · ส่งค่าเข้า SQL อย่างปลอดภัย · function ดึงข้อมูลทีละวัน**

ต้องเปิด Docker ไว้ (`docker compose up -d` ดู README)

## 3.1 DataFrame: ตารางใน Python

`import` = หยิบเครื่องมือที่มีคนเขียนไว้แล้วมาใช้ (บทที่ 4 อธิบายละเอียด) · `pd` คือชื่อเล่นที่ทุกคนใช้เรียก pandas

**DataFrame** = ตาราง (แถว × คอลัมน์) · list ของ dict (บทที่ 1) แปลงเป็นตารางได้ทันที: **1 dict = 1 แถว**

In [ ]:
import pandas as pd

orders = [
    {"order_id": 1, "channel": "web", "amount": 350.0},
    {"order_id": 2, "channel": "ios", "amount": 1290.0},
    {"order_id": 3, "channel": "web", "amount": 89.0},
]
df = pd.DataFrame(orders)
print(df.shape)              # (จำนวนแถว, จำนวนคอลัมน์)
print(df.dtypes)             # ชนิดของแต่ละคอลัมน์
df                           # บรรทัดสุดท้ายของ cell = แสดงเป็นตารางสวย ๆ

In [ ]:
print("ยอดรวม:", df["amount"].sum())          # df["ชื่อคอลัมน์"] = 1 คอลัมน์
print("ช่องทาง:", df["channel"].unique().tolist())

## 3.2 อ่านไฟล์ CSV

In [ ]:
mkt = pd.read_csv("../data/marketplace/marketplace_orders_2026-09-20.csv")
print(mkt.shape)
mkt.head()                   # 5 แถวแรก

In [ ]:
# อ่านหลายไฟล์: หาไฟล์ -> อ่านทีละไฟล์ -> ต่อกัน
from pathlib import Path

files = sorted(Path("../data/marketplace").glob("marketplace_orders_2026-09-2*.csv"))   # * = อะไรก็ได้
tables = []
for f in files:
    one_day = pd.read_csv(f)
    one_day["source_file"] = f.name              # จดไว้ว่ามาจากไฟล์ไหน (ตรวจย้อนหลังได้)
    tables.append(one_day)

all_mkt = pd.concat(tables, ignore_index=True)
print(len(files), "ไฟล์ ->", all_mkt.shape)
all_mkt["source_file"].value_counts().sort_index()

## 3.3 เลือกคอลัมน์ · กรองแถว · เพิ่มคอลัมน์

In [ ]:
all_mkt[["marketplace", "sku", "qty", "unit_price"]].head(3)          # เลือกบางคอลัมน์: ใส่ list ของชื่อ

In [ ]:
completed = all_mkt[all_mkt["status"] == "completed"]                 # กรองแถว
big_orders = all_mkt[(all_mkt["qty"] >= 2) & (all_mkt["status"] != "cancelled")]   # หลายเงื่อนไข: & และ, | หรือ
print(len(all_mkt), len(completed), len(big_orders))

In [ ]:
all_mkt["order_datetime"] = pd.to_datetime(all_mkt["order_datetime"])     # ข้อความ -> วันเวลา
all_mkt["order_date"] = all_mkt["order_datetime"].dt.date
all_mkt["line_total"] = all_mkt["qty"] * all_mkt["unit_price"]            # คำนวณคอลัมน์ใหม่ทั้งคอลัมน์ในบรรทัดเดียว
all_mkt[["order_datetime", "order_date", "qty", "unit_price", "line_total"]].head()

In [ ]:
# หาข้อมูลเสีย: จำนวน <= 0 (ไฟล์วันที่ 26 มีแถวแบบนี้)
all_mkt[all_mkt["qty"] <= 0][["source_file", "order_ref", "sku", "qty"]]

## 3.4 สรุปด้วย `groupby` (เหมือน GROUP BY ใน SQL)

In [ ]:
good = all_mkt[(all_mkt["status"] != "cancelled") & (all_mkt["qty"] > 0)]
mkt_daily = (good.groupby(["order_date", "marketplace"], as_index=False)
                 .agg(orders=("order_ref", "nunique"), units=("qty", "sum"), revenue=("line_total", "sum"))
                 .sort_values(["order_date", "revenue"], ascending=[True, False]))
mkt_daily.head(6)

`agg(ชื่อคอลัมน์ใหม่=("คอลัมน์เดิม", "วิธีสรุป"))` วิธีสรุป: `"sum"`, `"count"`, `"nunique"` (นับไม่ซ้ำ), `"mean"`, `"max"`

💡 `.agg(...)` รับ **keyword argument** แบบที่เรียนในบทที่ 2: ชื่อ = ค่า

## 3.5 ต่อฐานข้อมูล PostgreSQL

การต่อฐานข้อมูลต้องรู้ 5 อย่าง: **host, port, ชื่อฐานข้อมูล, user, password**
เก็บไว้ในไฟล์ `.env` (ไม่ขึ้น git) แล้วให้ Python อ่าน (บทที่ 6 อธิบายละเอียดว่าทำไม)

ผู้ใช้ `student` **อ่านได้อย่างเดียว** (ฐานข้อมูลจริงของร้าน ห้ามเขียนกลับ) เขียนได้เฉพาะ schema `lab`

In [ ]:
import os

import psycopg
from dotenv import load_dotenv

load_dotenv()                                  # อ่านไฟล์ .env
DB = {
    "host": os.getenv("POSTGRES_HOST", "127.0.0.1"),
    "port": os.getenv("POSTGRES_PORT", "5432"),
    "dbname": os.getenv("POSTGRES_DB", "ecommerce"),
    "user": os.getenv("POSTGRES_USER", "student"),
    "password": os.getenv("POSTGRES_PASSWORD", "student123"),
}

with psycopg.connect(**DB) as conn:            # ** แตก dict เป็น host=..., port=... (บทที่ 2.4)
    row = conn.execute("SELECT count(*), min(ordered_at), max(ordered_at) FROM orders").fetchone()
print(row)
print(f"มี order {row[0]:,} รายการ ตั้งแต่ {row[1]} ถึง {row[2]}")

- `with ... as conn:` = **เปิดการเชื่อมต่อ ใช้ แล้วปิดให้เองเมื่อจบย่อหน้า** แม้เกิด error (บทที่ 4)
- `.fetchone()` = แถวแรก (tuple: เข้าถึงด้วยตำแหน่ง `row[0]`) · `.fetchall()` = ทุกแถว (list ของ tuple)

In [ ]:
with psycopg.connect(**DB) as conn:
    rows = conn.execute("SELECT sales_channel, count(*) FROM orders GROUP BY sales_channel").fetchall()
for channel, n in rows:                        # แตก tuple ออกเป็น 2 ตัวแปร
    print(f"{channel:8s} {n:>8,}")

## 3.6 ส่งค่าเข้า SQL อย่างปลอดภัย: `%(ชื่อ)s` + params

อยากได้ข้อมูลของวันที่ที่เลือก **อย่าเอาค่ามาต่อเป็นข้อความ SQL เอง** (f-string) ให้เขียนช่องว่าง `%(ชื่อ)s`
แล้วส่งค่าเป็น dict ไปแยกกัน ฐานข้อมูลจะใส่ค่าให้อย่างปลอดภัย

In [ ]:
sql = """
SELECT sales_channel, count(*) AS orders, sum(total_amount) AS revenue
FROM orders
WHERE ordered_at >= %(day)s::date
  AND ordered_at <  %(day)s::date + 1
  AND status <> 'cancelled'
GROUP BY sales_channel
ORDER BY sales_channel
"""
with psycopg.connect(**DB) as conn:
    rows = conn.execute(sql, {"day": "2026-09-20"}).fetchall()
rows                     # Decimal = ตัวเลขทศนิยมแบบแม่นยำ (ใช้กับเงิน) คำนวณได้ปกติ

- `ordered_at >= วันนั้น AND ordered_at < วันถัดไป` = **1 วันเต็ม ไม่ตกหล่นวินาทีสุดท้าย** (บทที่ 5)
- ⚠️ ทำไมห้ามต่อข้อความ: ถ้าค่ามาจากคนอื่น เช่น `day = "2026-09-20'; DELETE FROM orders; --"`
  แล้วเราเขียน `f"WHERE ordered_at >= '{day}'"` = เราสั่งลบข้อมูลเอง (เรียกว่า **SQL injection**)

## 3.7 ผลของ SQL → DataFrame ด้วย `pd.read_sql`

pandas ต่อฐานข้อมูลผ่าน **engine** (สร้างครั้งเดียว ใช้ได้ตลอด) ที่เขียนค่าการเชื่อมต่อเป็น URL บรรทัดเดียว

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg://{DB['user']}:{DB['password']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)
channel_sales = pd.read_sql(sql, engine, params={"day": "2026-09-20"})
channel_sales

In [ ]:
print(channel_sales.dtypes)
print("ยอดรวมวันที่ 20:", channel_sales["revenue"].sum())

## 3.8 ห่อเป็น function: ดึงข้อมูลทีละวัน

งานจริงเรา **ไม่ดึงทั้งตารางทุกครั้ง** (orders มีเป็นแสนแถว และเพิ่มขึ้นทุกวัน) แต่ดึงทีละวัน
function รับวันที่ → คืน DataFrame ของวันนั้น · เรียกกี่วันก็ได้

In [ ]:
def extract_orders(day: str) -> pd.DataFrame:
    """order ของวันนั้น (เวลาไทย) ทุกสถานะ"""
    query = """
        SELECT order_id, order_number, sales_channel, status, total_amount, ordered_at
        FROM orders
        WHERE ordered_at >= %(day)s::date AND ordered_at < %(day)s::date + 1
        ORDER BY ordered_at
    """
    return pd.read_sql(query, engine, params={"day": day})


day_orders = extract_orders("2026-09-24")
print(day_orders.shape)
day_orders.head()

In [ ]:
def summarize(df: pd.DataFrame) -> dict:
    """ตัวเลขสรุปสั้น ๆ ของ 1 วัน (คืน dict เล็ก ๆ ไม่ใช่ทั้งตาราง)"""
    sold = df[df["status"] != "cancelled"]
    return {"orders": len(sold), "cancelled": len(df) - len(sold), "revenue": float(sold["total_amount"].sum())}


for day in ["2026-09-22", "2026-09-23", "2026-09-24"]:
    print(day, summarize(extract_orders(day)))

💡 แบ่งงานเป็น function เล็ก ๆ ที่ **รับวันที่เข้าไป** : ดึง (extract) → สรุป (summarize) → บันทึก
ทดสอบทีละชิ้นได้ และรันวันไหนซ้ำก็ได้ แบบนี้คือหน้าตาของงาน data engineer ที่จะเจอต่อไปเรื่อย ๆ

## 3.9 รวม 2 ตาราง (merge): ยอดเว็บ/แอป + ยอด marketplace

`merge` = เอาสองตารางมาต่อกันด้วยคอลัมน์ที่ตรงกัน (เหมือน JOIN ใน SQL)

In [ ]:
shop_daily = pd.read_sql(
    """
    SELECT ordered_at::date AS order_date, sum(total_amount) AS shop_revenue
    FROM orders
    WHERE ordered_at >= %(start)s AND ordered_at < %(end)s AND status <> 'cancelled'
    GROUP BY 1
    ORDER BY 1
    """,
    engine,
    params={"start": "2026-09-20", "end": "2026-09-25"},
)
mkt_by_day = good.groupby("order_date", as_index=False).agg(marketplace_revenue=("line_total", "sum"))

report = shop_daily.merge(mkt_by_day, on="order_date", how="left")       # how="left" = เก็บทุกวันของตารางซ้าย
report["total_revenue"] = report["shop_revenue"] + report["marketplace_revenue"]
report

## 3.10 บันทึกผล: ไฟล์ และตารางในฐานข้อมูล

In [ ]:
Path("../work").mkdir(exist_ok=True)
report.to_csv("../work/daily_sales.csv", index=False)
report.to_parquet("../work/daily_sales.parquet", index=False)     # parquet: เล็ก เร็ว เก็บชนิดข้อมูลไว้ด้วย
pd.read_parquet("../work/daily_sales.parquet").dtypes

In [ ]:
# เขียนเป็นตารางใน schema lab (if_exists="replace" = มีอยู่แล้วให้แทนที่ รันซ้ำได้)
report.to_sql("daily_sales", engine, schema="lab", if_exists="replace", index=False)
pd.read_sql("SELECT * FROM lab.daily_sales ORDER BY order_date", engine)

In [ ]:
# ❌ ข้อมูลของร้าน (schema public) เราอ่านได้อย่างเดียว เขียนไม่ได้
report.to_sql("daily_sales", engine, schema="public", if_exists="replace", index=False)

อ่านบรรทัดสุดท้ายของ error: `permission denied for schema public` = ไม่มีสิทธิ์เขียน
ระบบต้นทางจริงก็เป็นแบบนี้: ทีมข้อมูล **อ่าน** จากระบบของร้าน แล้วไปเก็บผลไว้ที่ของตัวเอง

💡 ลองเปิด DBeaver ต่อ PostgreSQL: host `localhost` port `5432` database `ecommerce` user `student` / `student123`
แล้ว `SELECT * FROM lab.daily_sales`

## ✅ สรุปบทที่ 3

| เรื่อง | ตัวอย่าง |
|---|---|
| สร้างตาราง | `pd.DataFrame(list_of_dicts)`, `pd.read_csv(path)`, `pd.concat([...])` |
| จัดการตาราง | `df[["a", "b"]]`, `df[df["x"] > 0]`, `df["new"] = df["a"] * df["b"]` |
| สรุป / รวม | `.groupby(...).agg(ชื่อ=("คอลัมน์", "sum"))`, `.merge(other, on=..., how="left")` |
| ต่อ PostgreSQL | `with psycopg.connect(**DB) as conn:` → `conn.execute(sql, params).fetchall()` |
| ส่งค่าเข้า SQL | `%(day)s` ใน SQL + `{"day": "2026-09-20"}` (ห้ามต่อข้อความเอง) |
| SQL → DataFrame | `pd.read_sql(sql, engine, params={...})` |
| บันทึก | `.to_csv()`, `.to_parquet()`, `.to_sql("t", engine, schema="lab", if_exists="replace")` |

🧪 **แบบฝึกท้ายบท:** เขียน function `top_products(day: str, n: int = 5) -> pd.DataFrame`
คืนสินค้าขายดี n อันดับแรกของวันนั้น (คอลัมน์ `sku`, `product_name`, `units`, `revenue` เรียงจาก revenue มากไปน้อย)
จากตาราง `orders` JOIN `order_items` JOIN `products` ไม่นับ order ที่ `cancelled`
แล้วลองเรียก `top_products("2026-09-20")` และ `top_products(day="2026-09-21", n=3)`

ใบ้: `order_items` มีคอลัมน์ `order_id`, `product_id`, `quantity`, `line_total` · `products` มี `product_id`, `sku`, `product_name`
และ SQL ใส่ `LIMIT %(n)s` ได้

In [ ]:
# เขียนต่อตรงนี้